# Exercise 1 — The OHLCV Format

OHLCV (Open, High, Low, Close, Volume) is the universal format for price data. Each row is one time period — a day, hour, or minute. Understanding this structure is the foundation for everything in Section 7: indicators, backtesting, and trading signals.

In [ ]:
import pandas as pd, math, sqlite3, tempfile, os

OHLCV_COLS = ["Open", "High", "Low", "Close", "Volume"]

def _synthetic(ticker="TEST", period="1y", interval="1d", n=50):
    prices = [100.0 * (1 + 0.3 * math.sin(i * 2 * math.pi / n)) for i in range(n)]
    dates  = pd.date_range("2023-01-01", periods=n, freq="B")
    close  = pd.Series(prices, index=dates)
    return pd.DataFrame({
        "Open":   close.shift(1).fillna(close.iloc[0]),
        "High":   close * 1.01,
        "Low":    close * 0.99,
        "Close":  close,
        "Volume": pd.Series([1_000_000 + i * 1_000 for i in range(n)], index=dates),
    })

# ── Exercise: implement make_ohlcv ────────────────────────────────────────────
# Build a synthetic OHLCV DataFrame following these rules:
#   Index  : pd.DatetimeIndex of n business days starting 2023-01-01
#   Close  : sine-wave starting at 100 with ±30% amplitude
#             100.0 * (1 + 0.3 * sin(i * 2π / n)) for row i
#   Open   : previous row's Close; first row = first Close
#   High   : Close * 1.01  (1% above close)
#   Low    : Close * 0.99  (1% below close)
#   Volume : 1_000_000 + i * 1_000 for row i

def make_ohlcv(n=20):
    """Build a synthetic OHLCV DataFrame with n rows.

    Args:
        n: number of trading-day rows

    Returns:
        pd.DataFrame with DatetimeIndex and columns Open, High, Low, Close, Volume
    """
    # TODO: compute prices using math.sin, build close Series with DatetimeIndex,
    # then construct the DataFrame with the five rules above.
    dates = pd.date_range("2023-01-01", periods=n, freq="B")
    return pd.DataFrame({
        "Open":   [100.0] * n,
        "High":   [101.0] * n,
        "Low":    [99.0]  * n,
        "Close":  [100.0] * n,
        "Volume": [1_000_000] * n,
    }, index=dates)


### Checks

In [ ]:
checks = 0

# 1 — returns a DataFrame with the 5 required columns
try:
    df = make_ohlcv(30)
    assert isinstance(df, pd.DataFrame)
    assert set(OHLCV_COLS).issubset(set(df.columns)), f"missing: {set(OHLCV_COLS) - set(df.columns)}"
    checks += 1; print("✅ 1 make_ohlcv returns DataFrame with correct columns")
except Exception as e:
    print("❌ 1:", e)

# 2 — index is DatetimeIndex with exactly n rows
try:
    df = make_ohlcv(30)
    assert isinstance(df.index, pd.DatetimeIndex), "index is not DatetimeIndex"
    assert len(df) == 30, f"expected 30 rows, got {len(df)}"
    checks += 1; print("✅ 2 DatetimeIndex with correct length")
except Exception as e:
    print("❌ 2:", e)

# 3 — High >= Close >= Low for every row
try:
    df = make_ohlcv(50)
    assert (df["High"] >= df["Close"]).all(), "High < Close in some rows"
    assert (df["Close"] >= df["Low"]).all(),  "Close < Low in some rows"
    checks += 1; print("✅ 3 High >= Close >= Low for all rows")
except Exception as e:
    print("❌ 3:", e)

# 4 — Volume increases by exactly 1000 per row
try:
    df = make_ohlcv(10)
    vol   = df["Volume"].tolist()
    diffs = [vol[i + 1] - vol[i] for i in range(len(vol) - 1)]
    assert all(abs(d - 1000) < 1e-6 for d in diffs), f"diffs={diffs}"
    checks += 1; print("✅ 4 Volume increases by 1000 per row")
except Exception as e:
    print("❌ 4:", e)

# 5 — Open equals previous Close (lag-1 relationship)
try:
    df = make_ohlcv(30)
    for i in range(1, len(df)):
        diff = abs(df["Open"].iloc[i] - df["Close"].iloc[i - 1])
        assert diff < 1e-9, f"row {i}: Open={df['Open'].iloc[i]}, prev Close={df['Close'].iloc[i-1]}"
    checks += 1; print("✅ 5 Open equals previous Close (lag-1 relationship)")
except Exception as e:
    print("❌ 5:", e)

print(f"\n{checks}/5 checks passed!")
